In [3]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Run this cell to setup imports from src correctly
import sys
import os
project_root = os.path.abspath(os.path.join(".."))  
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [12]:
%load_ext autoreload
%autoreload 2
from src.startup.start_docker_rabbitmq import start_rabbitmq
from src.communication.protocol import *
import src.communication.factory as factory
from src.communication.rabbitmq import *
import numpy as np
import threading
import time

# define a callback function to handle received messages
def on_message_received(ch, method, properties, body):
    try:
        print("✓ State:")
        print(body)
    except Exception as e:
        print(f"✗ Failed to decode the message: {e}")


# start docker instance
print("Script starting rabbitmq")
start_rabbitmq()

fac = factory.RabbitMQFactory()
print("Creating cli")
client = fac.create_rabbitmq()
print("Creating sub")
subscriber = fac.create_rabbitmq()


def run_subscriber():
    print("Subscriber connect to server")
    subscriber.connect_to_server()
    print("subscriber subscribe")
    subscriber.subscribe(ROUTING_KEY_CTRL, on_message_callback=on_message_received)
    print("Subscriber start consuming")
    subscriber.start_consuming()

threading.Thread(
    target=run_subscriber,
    daemon=True
).start()


# Let subscriber thread start and connect
time.sleep(2)

# Construct control message for loading a program
position = [0.0, -np.pi/2, np.pi/2, -np.pi/2, -np.pi/2, 0.0]
vel = 60 # deg/s
acc = 80 # deg/s²

msg = {
    CtrlMsgKeys.TYPE: CtrlMsgFields.LOAD_PROGRAM,
    CtrlMsgKeys.JOINT_POSITIONS: [position],
    CtrlMsgKeys.MAX_VELOCITY: vel,
    CtrlMsgKeys.ACCELERATION: acc,
}

# Create sender and that
print("Connecting client to rabbitmq")
client.connect_to_server()
print("Client sending message")
client.send_message(ROUTING_KEY_CTRL, msg)

print("script done")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Script starting rabbitmq
Searching for container with the name: rabbitmq-server
Container status: exited
Log will be stored in: c:\Users\Admin\Desktop\uni\DigitalTwins\DT-robot-arm\testing\logs\rabbitmq.log
Running docker-compose command: docker compose up --detach --build
docker-compose successful.
Service is not ready yet. Attempts remaining:9
Service is not ready yet. Attempts remaining:8
Service is not ready yet. Attempts remaining:7
Service is not ready yet. Attempts remaining:6
Service is not ready yet. Attempts remaining:5
Service is not ready yet. Attempts remaining:4
Service is not ready yet. Attempts remaining:3
Service is not ready yet. Attempts remaining:2
RabbitMQ ready:
 {"management_version":"3.12.14","rates_mode":"basic","sample_retention_policies":{"global":[600,3600,28800,86400],"basic":[600,3600],"detailed":[600]},"exchange_types":[{"name":"direct","description":"AMQP direct excha